In [ ]:
from data_utils import ReviewDataset
from sequence_models import RNNClassifier

from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from time import perf_counter
from torch.utils.data import DataLoader

import html
import matplotlib.pyplot as plt
import numpy as np
import re
import pandas as pd
import seaborn as sns
import time
import torch
import torch.nn as nn

## Experiment 1 — Dataset inspection

- Coi mỗi review là một sequence.

In [ ]:
data_path = '../data/IMDB Dataset.csv'

data = pd.read_csv(data_path)

# Thống kê sơ bộ về dataset
print(f'Số sample: {len(data)}')
print(f'Số classes: {data['sentiment'].nunique()}')

class_distribution = pd.DataFrame({
    'Count': data['sentiment'].value_counts(),
    'Percentage (%)': data['sentiment'].value_counts(normalize=True) * 100
})
print('Phân phối class trên tập dataset: ')
print(class_distribution)

# Kiểm tra dữ liệu bị thiếu trên bộ dataset
missing_values = data[["review", "sentiment"]].isna().sum()

# Thống kê số review rỗng trong dataset
blank_text = data["review"].fillna("").str.strip().eq("").sum()
# Thống kê số nhãn rỗng trong dataset
blank_labels = data["sentiment"].fillna("").str.strip().eq("").sum()

print("Bảng thống kê bộ dữ liệu bị thiếu:")
print(missing_values)

print("Số text rỗng:", blank_text)
print("Số nhãn rỗng:", blank_labels)

# Kiểm tra review bị trùng trong dataset
duplicate_reviews = data.duplicated(
    subset=["review"],
    keep="first"
)
print("Số review trùng:", duplicate_reviews.sum())

# In ra 10 review bị trùng trong bộ dataset
duplicates = data[data.duplicated(
    subset=["review"],
    keep=False
)].sort_values("review")

display(duplicates.head(10))

# Thống kê số review bị trùng nhưng khác nhãn
conflicting_labels = (
    data.groupby("review")["sentiment"]
      .nunique()
      .gt(1)
)
print(
    "Số review trùng nhưng khác nhãn:",
    conflicting_labels.sum()
)

# Loại bỏ duplicates trong bộ dataset
data = data.drop_duplicates(subset=["review"]).reset_index(drop=True)

def clean_review(text):
    '''
    Làm sạch cột review:
    + Chuyển HTML entities về ký tự thông thường
    + Xóa HTML tags, như <br />
    + Chuẩn hóa khoảng trắng và chuyển văn bản về chữ thường
    '''
    text = html.unescape(str(text))
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'\s+', ' ', text)

    return text.strip().lower()

def tokenize_review(text):
    """
    Chuyển review thành danh sách token.
    """
    text = clean_review(text)

    return re.findall(
        r"[A-Za-z0-9]+(?:[-'][A-Za-z0-9]+)*",
        text
    )

# Cleaning
data["cleaned_review"] = (
    data["review"]
    .fillna("")
    .apply(clean_review)
)

#Tokenization
data["tokens"] = data["cleaned_review"].apply(tokenize_review)

# Sequence length
data["document_length"] = data["tokens"].apply(len)

# Thống kê độ dài document chứa trong dataset, ở đây document length = sequence length
lengths = data["document_length"]

print('Thống kê về document length trong dataset: ')
print(f"Number of sequences: {len(data["document_length"])}")
print(f"Min:    {lengths.min()}")
print(f"Mean:   {lengths.mean():.2f}")
print(f"Median: {lengths.median():.2f}")
print(f"Max:    {lengths.max()}")
print(f"P90:    {np.percentile(lengths, 90):.2f}")
print(f"P95:    {np.percentile(lengths, 95):.2f}")
# Vẽ phân phối document length theo số review
plt.figure(figsize=(8, 5))

plt.hist(
    data["document_length"],
    bins=50,
    edgecolor="black"
)

plt.title("Phân phối độ dài review")
plt.xlabel("Sequence length")
plt.ylabel("Số review")
plt.tight_layout()
plt.show()

Số sample: 50000
Số classes: 2
Phân phối class trên tập dataset: 
           Count  Percentage (%)
sentiment                       
positive   25000            50.0
negative   25000            50.0
Bảng thống kê bộ dữ liệu bị thiếu:
review       0
sentiment    0
dtype: int64
Số text rỗng: 0
Số nhãn rỗng: 0
Số review trùng: 418


,review,sentiment
34058,"""Go Fish"" garnered Rose Troche rightly or wron...",negative
47467,"""Go Fish"" garnered Rose Troche rightly or wron...",negative
29956,"""Three"" is a seriously dumb shipwreck movie. M...",negative
31488,"""Three"" is a seriously dumb shipwreck movie. M...",negative
47527,"""Witchery"" might just be the most incoherent a...",negative
2976,"""Witchery"" might just be the most incoherent a...",negative
7949,'Dead Letter Office' is a low-budget film abou...,negative
32260,'Dead Letter Office' is a low-budget film abou...,negative
18022,(Spoilers)<br /><br />Oh sure it's based on Mo...,negative
22449,(Spoilers)<br /><br />Oh sure it's based on Mo...,negative


Số review trùng nhưng khác nhãn: 0


Nếu phải chọn `max_length`, nên chọn dựa theo thông tin về phân phối sequence length (min, mean, median, max, p90, p95), lượng thông tin bị mất trong mỗi review và hiệu năng mô hình

## Experiment 2 — Padding và truncation

In [ ]:
data = data[
    ['review', 'tokens', 'document_length', 'sentiment']
].copy()

train_data, temp_data = train_test_split(
    data,
    test_size = 0.4,
    random_state = 42,
    stratify = data['sentiment']
)

val_data, test_data = train_test_split(
    temp_data,
    test_size = 0.5,
    random_state = 42,
    stratify = temp_data['sentiment']
)

# Kiểm tra phân phối nhãn
for name, dataset in [
    ("Train", train_data),
    ("Validation", val_data),
    ("Test", test_data)
]:
    print(f"\n{name}: {len(dataset)} samples")
    print(dataset["sentiment"].value_counts(normalize=True))

max_lengths = [50, 100, 200]

sequence_lengths = np.array([data['document_length']])

total_tokens = sequence_lengths.sum()

token_retained_rate = []

for max_length in max_lengths:
    retained_tokens = np.minimum(
        sequence_lengths, max_length
    ).sum()

    retained_pct = (retained_tokens / total_tokens) * 100

    token_retained_rate.append(retained_pct)


Train: 29749 samples
sentiment
positive    0.501866
negative    0.498134
Name: proportion, dtype: float64

Validation: 9916 samples
sentiment
positive    0.501916
negative    0.498084
Name: proportion, dtype: float64

Test: 9917 samples
sentiment
positive    0.501865
negative    0.498135
Name: proportion, dtype: float64


In [ ]:
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"

PAD_ID = 0
UNK_ID = 1

MAX_VOCAB_SIZE = 20000
MIN_FREQ = 2

# Đếm tần suất token trong train set
counter = Counter(
    token
    for tokens in train_data["tokens"]
    for token in tokens
)

# Khởi tạo vocabulary
vocab = {
    PAD_TOKEN: PAD_ID,
    UNK_TOKEN: UNK_ID
}

# Bổ sung các token phổ biến
for token, freq in counter.most_common():
    if freq < MIN_FREQ:
        break

    if len(vocab) >= MAX_VOCAB_SIZE:
        break

    vocab[token] = len(vocab)

print(f"Vocabulary size: {len(vocab)}")

Vocabulary size: 20000


In [ ]:
MAX_LENGTH = 50
BATCH_SIZE = 64

train_dataset = ReviewDataset(
    train_data, vocab, MAX_LENGTH
)

print("Sample:", train_dataset.samples[0])
print("Type of input_ids:", type(train_dataset.samples[0][0]))
print("Type of length:", type(train_dataset.samples[0][1]))
print("Type of label:", type(train_dataset.samples[0][2]))

train_loader = DataLoader(
    train_dataset,
    batch_size = BATCH_SIZE,
    shuffle = True
)

input_ids, lengths, labels = next(iter(train_loader))

print('Input IDs: ', input_ids.shape)
print('Lengths: ', lengths.shape)
print('Labels: ', labels.shape)
print('Min actual length: ', lengths.min().item())
print('Max actual length: ', lengths.max().item())

Sample: ([70, 21, 156, 11, 17, 7, 52, 420, 153, 413, 4, 210, 9, 15, 2, 1002, 238, 313, 4, 66, 6, 357, 2, 10, 102, 46, 229, 27, 137, 8, 54, 825, 2559, 2, 17, 7, 3, 79, 2123, 5, 202, 4, 776, 832, 1232, 5404, 7, 888, 14, 3], 50, 1.0)
Type of input_ids: <class 'list'>
Type of length: <class 'int'>
Type of label: <class 'float'>
Input IDs:  torch.Size([64, 50])
Lengths:  torch.Size([64])
Labels:  torch.Size([64])
Min actual length:  50
Max actual length:  50


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    total_loss = 0.0
    total_samples = 0

    for input_ids, lengths, labels in loader:
        input_ids = input_ids.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model(input_ids, lengths)

        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size
        total_samples += batch_size

    return total_loss / total_samples

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    total_samples = 0

    y_true = []
    y_pred = []

    for input_ids, lengths, labels in loader:
        input_ids = input_ids.to(device)
        labels = labels.to(device)

        logits = model(input_ids, lengths)
        loss = criterion(logits, labels)

        predictions = (logits >= 0).long()

        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size
        total_samples += batch_size

        y_true.extend(labels.int().cpu().tolist())
        y_pred.extend(predictions.cpu().tolist())

    return {
        'loss': total_loss / total_samples,
        'accuracy': accuracy_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred)
    }

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Hyperparameters cố định
BATCH_SIZE = 64
EMBEDDING_DIM = 100
HIDDEN_DIM = 128
LEARNING_RATE = 0.001
EPOCHS = 5

results = []
histories = {}
trained_models = {}

for max_length, retained_pct in zip(
    max_lengths, token_retained_rate
):
    print(f"\n========== MAX LENGTH = {max_length} ==========")

    # 1. Dataset
    train_dataset = ReviewDataset(
        train_data, vocab, max_length
    )
    val_dataset = ReviewDataset(
        val_data, vocab, max_length
    )

    # 2. DataLoader
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    # 3. Khởi tạo model mới
    torch.manual_seed(42)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(42)

    model = RNNClassifier(
        vocab_size=len(vocab),
        embedding_dim=EMBEDDING_DIM,
        hidden_dim=HIDDEN_DIM,
        pad_id=vocab["<PAD>"]
    ).to(device)

    # 4. Loss và Optimizer
    criterion = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE
    )

    # 5. Training và Validation
    history = {
        "train_loss": [],
        "val_loss": [],
        "val_accuracy": [],
        "val_f1": []
    }

    training_time = 0.0

    for epoch in range(EPOCHS):
        if device.type == "cuda":
            torch.cuda.synchronize()

        start = perf_counter()

        train_loss = train_one_epoch(
            model, train_loader,
            criterion, optimizer, device
        )

        if device.type == "cuda":
            torch.cuda.synchronize()

        training_time += perf_counter() - start

        val_metrics = evaluate(
            model, val_loader, criterion, device
        )

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_metrics["loss"])
        history["val_accuracy"].append(val_metrics["accuracy"])
        history["val_f1"].append(val_metrics["f1"])

        print(
            f"Epoch {epoch + 1}/{EPOCHS} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_metrics['loss']:.4f} | "
            f"Val F1: {val_metrics['f1']:.4f}"
        )

    # 6. Lưu kết quả
    results.append({
        "Max length": max_length,
        "% tokens retained": retained_pct,
        "Training time": training_time,
        "Accuracy": val_metrics["accuracy"],
        "F1": val_metrics["f1"]
    })

    histories[max_length] = history
    trained_models[max_length] = model

# 7. Tổng hợp kết quả
results_df = pd.DataFrame(results)

display(results_df)


========== MAX LENGTH = 50 ==========
Epoch 1/5 | Train Loss: 0.6959 | Val Loss: 0.6900 | Val F1: 0.5662
Epoch 2/5 | Train Loss: 0.6898 | Val Loss: 0.6897 | Val F1: 0.4981
Epoch 3/5 | Train Loss: 0.6765 | Val Loss: 0.6928 | Val F1: 0.5870
Epoch 4/5 | Train Loss: 0.6569 | Val Loss: 0.6886 | Val F1: 0.5397
Epoch 5/5 | Train Loss: 0.6355 | Val Loss: 0.7986 | Val F1: 0.5357

========== MAX LENGTH = 100 ==========
Epoch 1/5 | Train Loss: 0.6927 | Val Loss: 0.6772 | Val F1: 0.6833
Epoch 2/5 | Train Loss: 0.6810 | Val Loss: 0.6904 | Val F1: 0.5336
Epoch 3/5 | Train Loss: 0.6573 | Val Loss: 0.6792 | Val F1: 0.5641
Epoch 4/5 | Train Loss: 0.6284 | Val Loss: 0.6535 | Val F1: 0.5682
Epoch 5/5 | Train Loss: 0.6020 | Val Loss: 0.6909 | Val F1: 0.5663

========== MAX LENGTH = 200 ==========
Epoch 1/5 | Train Loss: 0.6786 | Val Loss: 0.6523 | Val F1: 0.5732
Epoch 2/5 | Train Loss: 0.6557 | Val Loss: 0.6194 | Val F1: 0.6387
Epoch 3/5 | Train Loss: 0.6129 | Val Loss: 0.6166 | Val F1: 0.6325
Epoch 4/5 

,Max length,% tokens retained,Training time,Accuracy,F1
0,50,21.695197,493.886820,0.590258,0.535710
1,100,41.887034,1120.900259,0.572307,0.566316
2,200,69.000406,2745.856859,0.634833,0.642158


In [ ]:
# Đánh giá lại các mô hình RNN đã huấn luyện
criterion = nn.BCEWithLogitsLoss()

for max_length in max_lengths:
    val_dataset = ReviewDataset(
        val_data, vocab, max_length
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    metrics = evaluate(
        trained_models[max_length],
        val_loader,
        criterion,
        device
    )

    # Cập nhật kết quả đã lưu
    for result in results:
        if result["Max length"] == max_length:
            result["Accuracy"] = metrics["accuracy"]
            result["F1"] = metrics["f1"]
            break

results_df = pd.DataFrame(results)
display(results_df)


,Max length,% tokens retained,Training time,Accuracy,F1
0,50,21.695197,493.886820,0.590258,0.535710
1,100,41.887034,1120.900259,0.572307,0.566316
2,200,69.000406,2745.856859,0.634833,0.642158


- Khi tăng `max_length` thì chưa chắc cải thiện được hiệu năng model.
- Nếu `max_length` quá nhỏ thì các thông tin có thể mất là các từ có hàm ý phủ định, các từ chuyển ý nghĩa văn bản và các từ biểu thị cảm xúc.

## Core implementation — kiểm tra SimpleRNNCell

`rnn_cell.py` cài đặt một bước $h_t=\tanh(W_{xh}x_t+W_{hh}h_{t-1}+b)$.
Kiểm tra cùng trọng số và input với `torch.nn.RNNCell` để xác nhận phép tính.

In [ ]:
from rnn_cell import SimpleRNNCell

rng = np.random.default_rng(42)
manual_cell = SimpleRNNCell(input_size=3, hidden_size=4)
manual_cell.W_xh = rng.normal(size=(4, 3)).astype(np.float32)
manual_cell.W_hh = rng.normal(size=(4, 4)).astype(np.float32)
manual_cell.b_h = rng.normal(size=(4, 1)).astype(np.float32)
x_t = rng.normal(size=(3, 2)).astype(np.float32)
h_prev = rng.normal(size=(4, 2)).astype(np.float32)
manual_output = manual_cell.forward(x_t, h_prev)

torch_cell = nn.RNNCell(input_size=3, hidden_size=4)
with torch.no_grad():
    torch_cell.weight_ih.copy_(torch.from_numpy(manual_cell.W_xh))
    torch_cell.weight_hh.copy_(torch.from_numpy(manual_cell.W_hh))
    torch_cell.bias_ih.copy_(torch.from_numpy(manual_cell.b_h[:, 0]))
    torch_cell.bias_hh.zero_()
    torch_output = torch_cell(torch.from_numpy(x_t.T), torch.from_numpy(h_prev.T))

max_difference = np.max(np.abs(manual_output.T - torch_output.numpy()))
print(f'Max difference vs nn.RNNCell: {max_difference:.2e}')
assert np.allclose(manual_output.T, torch_output.numpy(), atol=1e-6)

# Experiment 3: Baseline

In [9]:
# Chuyển danh sách token thành chuỗi
X_train = train_data["tokens"].apply(" ".join)
X_val = val_data["tokens"].apply(" ".join)
X_test = test_data["tokens"].apply(" ".join)

# Chuyển nhãn về 0/1
label_map = {
    "negative": 0,
    "positive": 1,
}

y_train = train_data["sentiment"].map(label_map)
y_val = val_data["sentiment"].map(label_map)
y_test = test_data["sentiment"].map(label_map)

In [10]:
baseline_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            tokenizer=str.split,
            token_pattern=None,
            lowercase=False,
            max_features=20000,
            ngram_range=(1, 1)
        )
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )
])

start_time = time.perf_counter()

baseline_model.fit(X_train, y_train)

baseline_training_time = (
    time.perf_counter() - start_time
)

# Đánh giá trên validation set
y_pred_baseline = baseline_model.predict(X_val)

baseline_metrics = {
    "Model": "Logistic Regression",
    "Representation": "TF-IDF",
    "Accuracy": accuracy_score(y_val, y_pred_baseline),
    "Precision": precision_score(y_val, y_pred_baseline),
    "Recall": recall_score(y_val, y_pred_baseline),
    "F1": f1_score(y_val, y_pred_baseline),
    "Training time": baseline_training_time
}

baseline_metrics


{'Model': 'Logistic Regression',
 'Representation': 'TF-IDF',
 'Accuracy': 0.8915893505445744,
 'Precision': 0.8800155823918971,
 'Recall': 0.9077757685352622,
 'F1': 0.8936801503313223,
 'Training time': 17.933067100006156}

In [11]:
MAX_LENGTH = 200

rnn_model = trained_models[MAX_LENGTH]

val_dataset = ReviewDataset(
    val_data, vocab, MAX_LENGTH
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False
)

rnn_model.eval()

rnn_true = []
rnn_pred = []

with torch.no_grad():
    for input_ids, lengths, labels in val_loader:
        input_ids = input_ids.to(device)

        logits = rnn_model(input_ids, lengths)
        predictions = (logits >= 0).long()

        rnn_true.extend(labels.int().tolist())
        rnn_pred.extend(predictions.cpu().tolist())

rnn_metrics = {
    "Model": "Simple RNN",
    "Representation": "Embedding",
    "Accuracy": accuracy_score(rnn_true, rnn_pred),
    "Precision": precision_score(rnn_true, rnn_pred),
    "Recall": recall_score(rnn_true, rnn_pred),
    "F1": f1_score(rnn_true, rnn_pred),
    "Training time": results_df.loc[
        results_df["Max length"] == MAX_LENGTH,
        "Training time"
    ].iloc[-1]
}

In [12]:
comparison_df = pd.DataFrame([
    baseline_metrics,
    rnn_metrics
])

display(comparison_df)

,Model,Representation,Accuracy,Precision,Recall,F1,Training time
0,Logistic Regression,TF-IDF,0.891589,0.880016,0.907776,0.893680,17.933067
1,Simple RNN,Embedding,0.634833,0.631855,0.652803,0.642158,2745.856859


In [ ]:
from pathlib import Path
import torch

CHECKPOINT_DIR = Path("checkpoints")
CHECKPOINT_DIR.mkdir(exist_ok=True)

for max_length, model in trained_models.items():
    torch.save(
        model.state_dict(),
        CHECKPOINT_DIR / f"rnn_len{max_length}.pt"
    )

print("Đã lưu các mô hình RNN.")

# Experiment 4: RNN vs LSTM vs GRU

In [13]:
from torch.nn.utils.rnn import pack_padded_sequence

class SequenceClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim=100, hidden_dim=128, pad_id=0, architecture='RNN'):
        super().__init__()
        layers = {'RNN': nn.RNN, 'LSTM': nn.LSTM, 'GRU': nn.GRU}
        self.architecture = architecture.upper()
        if self.architecture not in layers:
            raise ValueError(f'Unknown architecture: {architecture}')
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_id)
        self.recurrent = layers[self.architecture](embedding_dim, hidden_dim, batch_first=True)
        self.classifier = nn.Linear(hidden_dim, 1)

    def forward(self, input_ids, lengths):
        embedded = self.embedding(input_ids)
        packed = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, hidden = self.recurrent(packed)
        if self.architecture == 'LSTM':
            hidden = hidden[0]
        return self.classifier(hidden[-1]).squeeze(1)


def make_loaders(max_length, batch_size=BATCH_SIZE):
    datasets = {name: ReviewDataset(frame, vocab, max_length) for name, frame in [('train', train_data), ('val', val_data), ('test', test_data)]}
    return {name: DataLoader(dataset, batch_size=batch_size, shuffle=(name == 'train')) for name, dataset in datasets.items()}


def fit_sequence_model(model, loaders, epochs=EPOCHS):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    history = {'train_loss': [], 'val_loss': [], 'val_accuracy': [], 'val_f1': []}
    best_state, best_loss = None, float('inf')
    start = perf_counter()
    for _ in range(epochs):
        history['train_loss'].append(train_one_epoch(model, loaders['train'], criterion, optimizer, device))
        metrics = evaluate(model, loaders['val'], criterion, device)
        for key, value in [('val_loss', metrics['loss']), ('val_accuracy', metrics['accuracy']), ('val_f1', metrics['f1'])]:
            history[key].append(value)
        if metrics['loss'] < best_loss:
            best_loss = metrics['loss']
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
    if best_state is not None:
        model.load_state_dict(best_state)
    return history, perf_counter() - start


@torch.no_grad()
def evaluate_all(model, loader):
    criterion = nn.BCEWithLogitsLoss()
    model.eval(); true, pred, total_loss, total = [], [], 0.0, 0
    for input_ids, lengths, labels in loader:
        logits = model(input_ids.to(device), lengths)
        labels = labels.to(device)
        total_loss += criterion(logits, labels).item() * labels.size(0); total += labels.size(0)
        true.extend(labels.int().cpu().tolist()); pred.extend((logits >= 0).long().cpu().tolist())
    return {'Loss': total_loss / total, 'Accuracy': accuracy_score(true, pred), 'Precision': precision_score(true, pred, zero_division=0), 'Recall': recall_score(true, pred, zero_division=0), 'F1': f1_score(true, pred, zero_division=0)}


In [ ]:
EXP4_MAX_LENGTH = 100
architecture_results, architecture_histories, architecture_models = [], {}, {}
architecture_validation_results = []

for architecture in ['RNN', 'LSTM', 'GRU']:
    torch.manual_seed(42)
    loaders = make_loaders(EXP4_MAX_LENGTH)
    model = SequenceClassifier(
        len(vocab), EMBEDDING_DIM, HIDDEN_DIM, vocab[PAD_TOKEN], architecture
    ).to(device)
    history, training_time = fit_sequence_model(model, loaders)
    val_metrics = evaluate_all(model, loaders['val'])
    test_metrics = evaluate_all(model, loaders['test'])
    architecture_validation_results.append({
        'Model': architecture,
        'Validation loss': val_metrics['Loss'],
        'Validation F1': val_metrics['F1'],
        'Best epoch': int(np.argmin(history['val_loss'])) + 1,
    })
    architecture_results.append({
        'Model': architecture,
        'Parameters': sum(p.numel() for p in model.parameters()),
        'Training time': training_time,
        'Epochs': EPOCHS,
        'Sequence length': EXP4_MAX_LENGTH,
        'Vocabulary size': len(vocab),
        **test_metrics,
    })
    architecture_histories[architecture] = history
    architecture_models[architecture] = model

architecture_validation_results_df = pd.DataFrame(architecture_validation_results)
architecture_results_df = pd.DataFrame(architecture_results)
best_architecture = architecture_validation_results_df.sort_values(
    ['Validation F1', 'Validation loss'], ascending=[False, True]
).iloc[0]['Model']
print(f'Model selected by validation F1: {best_architecture}')
display(architecture_validation_results_df)
display(architecture_results_df)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for name, history in architecture_histories.items():
    epochs = range(1, len(history['train_loss']) + 1)
    axes[0].plot(epochs, history['train_loss'], label=f'{name} train')
    axes[0].plot(epochs, history['val_loss'], '--', label=f'{name} val')
    axes[1].plot(epochs, history['val_f1'], label=name)
axes[0].set(title='Train and validation loss', xlabel='Epoch', ylabel='Loss')
axes[1].set(title='Validation F1', xlabel='Epoch', ylabel='F1')
axes[0].legend()
axes[1].legend()
plt.tight_layout()
plt.show()

Đối chiếu giả thuyết ở trên bằng bảng validation và bảng test. F1, Precision và Recall cần được đọc cùng Accuracy; chênh lệch trên một lần chạy không đủ để kết luận kiến trúc nào luôn tốt hơn. Cell sau in ra nhận xét dựa trên kết quả thực tế.

In [ ]:
val_f1 = architecture_validation_results_df.set_index('Model')['Validation F1']
test_f1 = architecture_results_df.set_index('Model')['F1']
for gated in ['LSTM', 'GRU']:
    relation = 'cao hơn' if val_f1[gated] > val_f1['RNN'] else 'không cao hơn'
    print(f'{gated}: validation F1 {val_f1[gated]:.4f} {relation} RNN ({val_f1["RNN"]:.4f}); '
          f'test F1={test_f1[gated]:.4f}.')
print('Kết luận chỉ áp dụng cho dataset, split, seed và cấu hình của lần chạy này.')

# Experiment 5: Sequence length

Giữ GRU và các siêu tham số cố định, chỉ thay đổi `max_length` trong 50, 100, 200. Báo cáo tỷ lệ token giữ lại, thời gian fit, validation loss của checkpoint được chọn và test F1. Peak GPU memory được đo khi có CUDA; cột bộ nhớ input chỉ là ước lượng kích thước tensor `input_ids` của một batch.

In [ ]:
sequence_length_results = []
for max_length in [50, 100, 200]:
    torch.manual_seed(42)
    loaders = make_loaders(max_length)
    model = SequenceClassifier(
        len(vocab), EMBEDDING_DIM, HIDDEN_DIM, vocab[PAD_TOKEN], 'GRU'
    ).to(device)
    if device.type == 'cuda':
        torch.cuda.reset_peak_memory_stats(device)
    history, training_time = fit_sequence_model(model, loaders)
    val_metrics = evaluate_all(model, loaders['val'])
    test_metrics = evaluate_all(model, loaders['test'])
    retained = 100 * np.minimum(
        data['document_length'].to_numpy(), max_length
    ).sum() / data['document_length'].sum()
    peak_gpu_mib = (
        torch.cuda.max_memory_allocated(device) / 2**20
        if device.type == 'cuda' else np.nan
    )
    sequence_length_results.append({
        'Max length': max_length,
        '% tokens retained': retained,
        'Training time': training_time,
        'Validation loss': val_metrics['Loss'],
        'Test F1': test_metrics['F1'],
        'Parameters': sum(p.numel() for p in model.parameters()),
        'Input IDs per batch (MiB)': BATCH_SIZE * max_length * 8 / 2**20,
        'Peak GPU allocated (MiB)': peak_gpu_mib,
    })

sequence_length_results_df = pd.DataFrame(sequence_length_results)
display(sequence_length_results_df)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.lineplot(data=sequence_length_results_df, x='Max length', y='Test F1', marker='o', ax=axes[0])
sns.lineplot(data=sequence_length_results_df, x='Max length', y='Training time', marker='o', ax=axes[1])
axes[0].set_title('Context length vs test F1')
axes[1].set_title('Context length vs training time')
plt.tight_layout()
plt.show()

### Đánh đổi giữa ngữ cảnh và chi phí

Tỷ lệ token giữ lại tăng theo `max_length`, còn thời gian và F1 cần đọc từ kết quả chạy thực tế. Không suy ra chênh lệch F1 là ổn định từ một seed. Cell sau tính thay đổi giữa các cấu hình, tránh ghi sẵn kết quả của notebook tham khảo.

In [ ]:
ordered_lengths = sequence_length_results_df.sort_values('Max length').reset_index(drop=True)
for previous, current in zip(ordered_lengths.iloc[:-1].to_dict('records'),
                             ordered_lengths.iloc[1:].to_dict('records')):
    print(f"{previous['Max length']} -> {current['Max length']}: "
          f"tokens retained {current['% tokens retained'] - previous['% tokens retained']:+.2f} pp, "
          f"fit time {current['Training time'] - previous['Training time']:+.2f} s, "
          f"test F1 {current['Test F1'] - previous['Test F1']:+.4f}.")

# Experiment 6: Long-range dependency

Dùng cùng một chuỗi token nền dài 100 vị trí, chỉ thay một vị trí bằng `excellent` hoặc `terrible`. So sánh đầu, giữa và cuối trên cả ba mô hình của Experiment 4. Vì đây là câu tổng hợp, kết quả là quan sát có kiểm soát về vị trí token, không phải bằng chứng đầy đủ về vanishing gradient.

In [ ]:
neutral_seed = [
    word for word in 'the movie has a story with many scenes and characters and actors'.split()
    if word in vocab
]
assert len(neutral_seed) >= 5, 'Không đủ token trung tính trong vocabulary.'
assert all(word in vocab for word in ['excellent', 'terrible']),     'Từ khóa sentiment phải có trong vocabulary để thí nghiệm có ý nghĩa.'

exp6_length = EXP4_MAX_LENGTH
base_tokens = [neutral_seed[i % len(neutral_seed)] for i in range(exp6_length)]
position_specs = {'beginning': 2, 'middle': exp6_length // 2, 'end': exp6_length - 3}
rows = []
for architecture, model in architecture_models.items():
    model.eval()
    with torch.no_grad():
        for position_name, position in position_specs.items():
            for sentiment, keyword in [('positive', 'excellent'), ('negative', 'terrible')]:
                tokens = base_tokens.copy()
                tokens[position] = keyword
                ids = [vocab[word] for word in tokens]
                input_ids = torch.tensor([ids], dtype=torch.long, device=device)
                lengths = torch.tensor([len(ids)], dtype=torch.long)
                logit = model(input_ids, lengths).item()
                rows.append({
                    'Model': architecture,
                    'Position': position_name,
                    'Token index': position,
                    'Distance to end': exp6_length - position - 1,
                    'Keyword': keyword,
                    'Keyword sentiment': sentiment,
                    'Predicted sentiment': 'positive' if logit >= 0 else 'negative',
                    'P(positive)': torch.sigmoid(torch.tensor(logit)).item(),
                })
long_range_results_df = pd.DataFrame(rows)
display(long_range_results_df)

### Quan sát và giải thích

So sánh `P(positive)` **trong cùng model và cùng keyword** giữa ba vị trí. Vì mọi token nền và độ dài được giữ nguyên, khác biệt dự đoán liên quan đến vị trí token trong phép thử này. Không suy rộng từ sáu câu tổng hợp ra mọi review.

Về lý thuyết, gradient truyền ngược qua nhiều bước recurrent chứa tích các Jacobian $\partial h_t/\partial h_{t-1}$. Tích này có thể co nhỏ, khiến tín hiệu từ token ở xa khó tác động tới cập nhật trọng số; gates của LSTM/GRU tạo thêm đường lưu thông tin. Bảng dự đoán ở trên **không đo gradient**, nên chỉ là quan sát thực nghiệm chứ không chứng minh cơ chế đó.

## 22. Evaluation — test metrics và confusion matrix

Bảng Experiment 4 đã báo cáo Accuracy, Precision, Recall, F1, số tham số, thời gian huấn luyện, epochs, độ dài chuỗi và vocabulary size. Ma trận nhầm lẫn dưới đây dùng đúng checkpoint đã được chọn theo validation loss, trên cùng test split.

In [ ]:
from sklearn.metrics import confusion_matrix

@torch.no_grad()
def predict_loader(model, loader):
    model.eval()
    actual, predicted, probabilities = [], [], []
    for input_ids, lengths, labels in loader:
        logits = model(input_ids.to(device), lengths)
        probs = torch.sigmoid(logits).cpu().numpy()
        actual.extend(labels.int().numpy().tolist())
        predicted.extend((probs >= 0.5).astype(int).tolist())
        probabilities.extend(probs.tolist())
    return np.asarray(actual), np.asarray(predicted), np.asarray(probabilities)

exp4_test_loader = DataLoader(
    ReviewDataset(test_data, vocab, EXP4_MAX_LENGTH),
    batch_size=BATCH_SIZE, shuffle=False
)
test_predictions = {}
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for axis, architecture in zip(axes, ['RNN', 'LSTM', 'GRU']):
    actual, predicted, probabilities = predict_loader(
        architecture_models[architecture], exp4_test_loader
    )
    test_predictions[architecture] = (actual, predicted, probabilities)
    matrix = confusion_matrix(actual, predicted, labels=[0, 1])
    sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues', ax=axis,
                xticklabels=['Negative', 'Positive'],
                yticklabels=['Negative', 'Positive'])
    axis.set(title=architecture, xlabel='Predicted', ylabel='Actual')
plt.tight_layout()
plt.show()

## 23. Learning curves

Biểu đồ loss và validation F1 của ba mô hình đã được vẽ ở Experiment 4. Bảng dưới tóm tắt epoch có validation loss thấp nhất và so sánh loss cuối với loss tốt nhất. Validation loss tăng sau epoch tốt nhất **có thể** là dấu hiệu overfitting; cần xem thêm nhiều seed hoặc lần chạy để kết luận chắc hơn.

In [ ]:
curve_rows = []
for architecture, history in architecture_histories.items():
    best_index = int(np.argmin(history['val_loss']))
    curve_rows.append({
        'Model': architecture,
        'Best epoch': best_index + 1,
        'Train loss at best epoch': history['train_loss'][best_index],
        'Best validation loss': history['val_loss'][best_index],
        'Final train loss': history['train_loss'][-1],
        'Final validation loss': history['val_loss'][-1],
        'Validation loss increase': history['val_loss'][-1] - history['val_loss'][best_index],
    })
curve_summary_df = pd.DataFrame(curve_rows)
display(curve_summary_df)

## 24. Error analysis

Chọn mô hình bằng validation F1, rồi lấy ba dự đoán đúng và tối đa bốn dự đoán sai trên test. Các dấu hiệu như phủ định, chuyển ý và truncation chỉ là gợi ý để đọc thủ công; bảng này không tự xác định nguyên nhân lỗi.

In [ ]:
actual, predicted, probabilities = test_predictions[best_architecture]
correct_indices = np.flatnonzero(actual == predicted)
error_indices = np.flatnonzero(actual != predicted)
rng_error = np.random.default_rng(42)
selected = [
    ('correct', index) for index in rng_error.choice(
        correct_indices, size=min(3, len(correct_indices)), replace=False
    )
] + [
    ('error', index) for index in rng_error.choice(
        error_indices, size=min(4, len(error_indices)), replace=False
    )
]

error_rows = []
for outcome, index in selected:
    sample = test_data.iloc[int(index)]
    tokens = sample['tokens']
    error_rows.append({
        'Outcome': outcome,
        'True': 'positive' if actual[index] else 'negative',
        'Predicted': 'positive' if predicted[index] else 'negative',
        'P(positive)': probabilities[index],
        'Sequence length': len(tokens),
        'Truncated': len(tokens) > EXP4_MAX_LENGTH,
        'Negation cue': any(word in tokens for word in ['not', 'never', 'no']),
        'Contrast cue': any(word in tokens for word in ['but', 'however', 'although']),
        'Review excerpt': sample['review'][:400],
    })
error_analysis_df = pd.DataFrame(error_rows)
display(error_analysis_df)

## 25. So sánh với Lab 04

| Thuộc tính | TF-IDF + Logistic Regression | Average Word2Vec | RNN/LSTM/GRU + Embedding |
|---|---|---|---|
| Biểu diễn | Vector thưa theo từ/ngram | Vector dày trung bình | Vector dày học được và hidden state |
| Trật tự từ | Unigram bỏ thứ tự; ngram giữ một phần thứ tự cục bộ | Bị mất khi lấy trung bình | Cập nhật theo thứ tự token |
| Ngữ cảnh | Trọng số cố định theo đặc trưng | Vector từ cố định | Hidden state phụ thuộc các token trước |
| Phụ thuộc xa | Hạn chế với đặc trưng cục bộ | Không mô hình hóa trực tiếp | Có khả năng học; vẫn khó với chuỗi dài |
| Chi phí | Fit và suy luận tương đối nhẹ | Chi phí tạo vector/trung bình | Cần tính recurrent theo các timestep |

Bảng nêu **khả năng của kiến trúc**, không chứng minh mô hình đã học được phủ định hay dependency dài trên dữ liệu này. Với mô hình chỉ dùng hidden state cuối, thông tin của review dài còn phải nén vào một vector; các timestep recurrent cũng hạn chế khả năng song song hóa. Hai hạn chế này là động lực nghiên cứu attention và Transformer.

## 26. Application — dự đoán review mới

Pipeline: review thô → cùng tokenizer đã dùng khi huấn luyện → vocabulary → truncation/padding → mô hình được chọn bằng validation → xác suất sentiment. Bốn mẫu bên dưới kiểm tra câu ngắn, câu dài, phủ định và chuyển ý; kết quả phải được đọc sau khi chạy mô hình thật.

In [ ]:
@torch.no_grad()
def predict_sentiment(review, model, max_length=EXP4_MAX_LENGTH):
    tokens = tokenize_review(review)
    if not tokens:
        raise ValueError('Review không có token sau tiền xử lý.')
    token_ids = [vocab.get(token, vocab[UNK_TOKEN]) for token in tokens[:max_length]]
    length = len(token_ids)
    token_ids += [vocab[PAD_TOKEN]] * (max_length - length)
    model.eval()
    logits = model(
        torch.tensor([token_ids], dtype=torch.long, device=device),
        torch.tensor([length], dtype=torch.long)
    )
    probability = torch.sigmoid(logits)[0].item()
    return {
        'Predicted': 'positive' if probability >= 0.5 else 'negative',
        'P(positive)': probability,
        'Tokens': len(tokens),
        'Tokens used': length,
        'Truncated': len(tokens) > max_length,
    }

long_review = (
    'The film follows several characters through many ordinary scenes. '
    'The story moves slowly and returns to the same events from different angles. '
) * 7 + 'The ending was excellent.'
application_samples = [
    ('short', 'I really enjoyed this movie.'),
    ('long', long_review),
    ('negation', 'I did not like this movie at all.'),
    ('contrast', 'The beginning was slow, but the ending was excellent.'),
]
application_results_df = pd.DataFrame([
    {'Type': kind, 'Review': review[:160],
     **predict_sentiment(review, architecture_models[best_architecture])}
    for kind, review in application_samples
])
display(application_results_df)

## Kết quả để nộp

Chạy các cell theo thứ tự rồi lưu các chỉ số tính được vào `Lab5/results.csv`. Không dùng số liệu cố định của notebook tham khảo; file CSV chỉ phản ánh lần chạy hiện tại.

In [ ]:
from pathlib import Path

records = []
for _, row in results_df.iterrows():
    for metric in ['% tokens retained', 'Training time', 'Accuracy', 'F1']:
        records.append({'experiment': 'exp2', 'setting': f"max_length={int(row['Max length'])}",
                        'metric': metric, 'value': float(row[metric])})
for metric in ['Accuracy', 'Precision', 'Recall', 'F1', 'Training time']:
    records.append({'experiment': 'exp3', 'setting': 'TF-IDF + Logistic Regression',
                    'metric': metric, 'value': float(baseline_metrics[metric])})
for _, row in architecture_results_df.iterrows():
    for metric in ['Parameters', 'Training time', 'Accuracy', 'Precision', 'Recall', 'F1']:
        records.append({'experiment': 'exp4', 'setting': row['Model'],
                        'metric': metric, 'value': float(row[metric])})
for _, row in sequence_length_results_df.iterrows():
    for metric in ['% tokens retained', 'Training time', 'Validation loss', 'Test F1']:
        records.append({'experiment': 'exp5', 'setting': f"max_length={int(row['Max length'])}",
                        'metric': metric, 'value': float(row[metric])})
export_df = pd.DataFrame(records)
results_path = Path(data_path).resolve().parent.parent / 'results.csv'
export_df.to_csv(results_path, index=False)
print(f'Saved {len(export_df)} measurements to {results_path}')